In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
from rag_helper import RAGBase

instructions: str = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    instructions=instructions,
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini",
)

In [11]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```


In [17]:
messages: list[dict] = [
    {
        "role": "user", 
        "content": "I just discovered the course. Can I join it?"
    },
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—usually you can, if the course is still open for enrollment.\n\nIf you want, I can help you figure out:\n- whether it’s still possible to join,\n- how to register,\n- and what to say if you need to ask the instructor.\n\nA simple message you could send is:\n\n“Hi, I just discovered this course and I’m interested in joining. Is it still possible to enroll?”\n\nIf you share the course name or where you found it, I can help you check the next steps.'

In [26]:
def search(query: str, **kwargs: dict) -> str:
    """Search the course for the query.

    Parameters
    ----------
    query: str
        The query to search the course for.

    **kwargs: dict
        Additional keyword arguments to pass to the search function.

    Returns
    -------
    str
        The search results.
    """
    boost_dict: dict[str, float] = {
        "question": kwargs.get("question", 3.0),
        "section": kwargs.get("section", 0.5),
    }
    filter_dict: dict[str, str] = {
        "course": kwargs.get("course", "llm-zoomcamp"),
    }

    return index.search(
        query,
        num_results=kwargs.get("num_results", 5),
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [16]:
search_tool: dict = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [21]:
len(response.output)

1

In [22]:
call = response.output[0]

In [23]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course enroll join after start eligibility FAQ"}', call_id='call_KaC4Sh7jaeHEuEoeG3ejtDoB', name='search', type='function_call', id='fc_03ea675ad5b9bedd006a31a282434c81929d71aca2e9ab7bd3', namespace=None, status='completed')

In [29]:
import json

args = json.loads(call.arguments)
print(json.dumps(args, indent=4))

{
    "query": "Can I join the course late discovered course enroll join after start eligibility FAQ"
}


In [31]:
results = search(**args)

In [36]:
results_json = json.dumps(results, indent=4)

In [38]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
}

In [39]:
messages.append(call)

In [40]:
messages.append(function_call_output)

Use message history with tool calls and tool call outputs:

In [44]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [45]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.


In [47]:
usage = response.usage
usage.input_tokens, usage.output_tokens, usage.total_tokens

(775, 31, 806)

In [ ]:
def calculate_gpt54mini_price(
    input_tokens: int, output_tokens: int
) -> dict[str, float]:
    """Calculate the cost of using GPT-5.4-mini.

    Parameters
    ----------
    input_tokens: int
        The number of input tokens.
    output_tokens: int
        The number of output tokens.

    Returns
    -------
    dict[str, float]
        The cost of using GPT-5.4-mini.
    """
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION: float = 0.15  # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION: float = 0.60  # $0.60 / 1M output tokens

    input_cost: float = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost: float = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost: float = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $",  round(result["total_cost"], 8))

Input Cost: $ 9.78e-05
Output Cost: $ 1.98e-05
Total Cost: $ 0.0001176
